In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# | tags: [parameters]
subject_id = "170"

In [ ]:
from IPython.display import display
from hamilton import telemetry

telemetry.disable_telemetry()

from hamilton import driver
from spectral.flows import preprocessing
from spectral.helpers import style_cfg


filter_params = {
    "l_freq": 1.0,
    "h_freq": 40.0,
    "h_trans_bandwidth": "auto",
    "fir_window": "hamming",
    "fir_design": "firwin",
    "phase": "zero",
    "picks": ["ecg", "eeg"],
}

# All pipeline parameters in one place. Values below are the flow's own defaults,
# written out explicitly so a run is self-documenting. Change here, not in the library.
inputs = {
    "subject_id": subject_id,
    "filter_params": filter_params,
    "fline": [50, 100],  # notch: line noise + harmonic
    # --- were hardcoded inside flows/preprocessing.py ---
    "resample_freq": 250.0,  # Hz, applied before filtering
    "resample_method": "polyphase",
    "crop_edge_s": 3.0,  # s trimmed from each end (filter edge artefacts)
    "psd_fmax": 60.0,  # Hz, upper limit of the diagnostic PSD plots
    # --- were defaults buried in annotation.run_pyprep_cleaning ---
    "pyprep_resample_freq": 125.0,  # Hz, downsample for RANSAC speed
    "pyprep_random_state": 1337,  # RANSAC seed -> bad-channel detection reproducibility
}

# Initialize Driver
dr = driver.Driver({}, preprocessing)

#  Display
dr.display_all_functions(orient="TB", custom_style_function=style_cfg)

In [ ]:
results = dr.execute(
    final_vars=[
        "raw_dropped",
        "plot_raw_psd",
        "report_with_psd",
        "raw_filtered",
        "raw_annotated_pyprep",
        "plot_filtered_psd",
        "report_saved_path",
    ],
    inputs=inputs,
)


In [ ]:
# --- what did this run actually do? ---
import json, subprocess, numpy as np, mne
from spectral.utils import load_config, ProjectPaths

raw = results["raw_annotated_pyprep"].iloc[0]
cfg = load_config()

# ground truth, measured off the output
good = mne.pick_types(raw.info, eeg=True, exclude="bads")
d = raw.get_data(picks=good, start=0, stop=int(10 * raw.info["sfreq"]))
ref_ratio = float(np.sqrt((d.mean(0) ** 2).mean()) / np.sqrt((d**2).mean()))

record = {
    "subject_id": subject_id,
    "git_sha": subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True
    ).stdout.strip(),
    "sfreq_hz": raw.info["sfreq"],
    "highpass_hz": raw.info["highpass"],
    "lowpass_hz": raw.info["lowpass"],
    "duration_s": round(float(raw.times[-1]), 1),
    "n_channels": len(raw.ch_names),
    "n_bads": len(raw.info["bads"]),
    "bads": raw.info["bads"],
    "avg_referenced": ref_ratio < 1e-6,  # <- would have caught the reference bug
    "declared_inputs": {k: v for k, v in inputs.items() if k != "subject_id"},
    "from_settings_toml": {
        "session": cfg["experiment"].get("session"),
        "montage": cfg["preprocessing"].get("montage"),
        "n_channels_removed": len(cfg["preprocessing"]["channels_to_remove"]),
    },
}
print(json.dumps(record, indent=2, default=str))

p = ProjectPaths(subject_id)
(p.preprocessed / f"sub-{p.subject_id}_provenance.json").write_text(
    json.dumps(record, indent=2, default=str)
)


# Plots

In [ ]:
fig = results["raw_dropped"].iloc[0].plot(scalings="auto", show=True, butterfly=True)

In [ ]:
standard_scalings = {
    "eeg": 60e-6,  # 40 µV (Good for clean brainwaves)
    "ecg": 500e-6,  # 500 µV (ECG is naturally much larger)
    "eog": 150e-6,
}
fig = (
    results["raw_filtered"]
    .iloc[0]
    .plot(scalings=standard_scalings, show=True, butterfly=True)
)

In [ ]:
display(results["plot_raw_psd"].iloc[0])

In [ ]:
display(results["plot_filtered_psd"].iloc[0])

In [ ]:
sensor_plot = results["raw_annotated_pyprep"].iloc[0].plot_sensors(show_names=True)
